In [1]:
from optionanalytics.data.yahoo import fetch_option_chain

In [2]:
chain = fetch_option_chain("AAPL", "2026-08-21")

In [3]:
type(chain)

optionanalytics.models.option.OptionChain

In [4]:
chain.expiry, chain.underlying, chain.quotes[0:2]

(datetime.date(2026, 8, 21),
 'AAPL',
 [OptionQuote(option_type=<OptionType.CALL: 'call'>, strike=110.0, bid=222.4, ask=225.7, last_price=203.3, volume=345, open_interest=368, last_trade_time=Timestamp('2026-07-10 16:13:39+0000', tz='UTC')),
  OptionQuote(option_type=<OptionType.CALL: 'call'>, strike=115.0, bid=211.05, ask=214.75, last_price=181.59, volume=116, open_interest=116, last_trade_time=Timestamp('2026-06-15 18:44:11+0000', tz='UTC'))])

In [5]:
from optionanalytics.cleaning.filters import clean_option_chain
print("chain length before cleaning:", len(chain.quotes))

chain length before cleaning: 134


In [6]:
clean_chain = clean_option_chain(chain)

In [7]:
len(clean_chain.quotes)

129

In [8]:
len(chain.quotes)

134

In [9]:
from optionanalytics.volatility.smile import build_smile
from optionanalytics.models.market import MarketData

market_data = MarketData(
    spot=333.74, 
    risk_free_rate=0.0418, 
    volatility=0.20 # placeholder, ignored by IV solver
)

In [10]:
import datetime
valuation_date = datetime.date(2026,7,18)
smile = build_smile(clean_chain, market_data, valuation_date)

ValueError: Failed to compute implied volatility.

In [11]:
from optionanalytics.pricing.implied_volatility import implied_volatility
from optionanalytics.models.option import EuropeanOption
for quote in clean_chain.quotes:
    option = EuropeanOption(option_type=quote.option_type, strike=quote.strike, maturity=34/365.0)
    try:
        iv = implied_volatility(
            price=quote.mid_price,
            option=option,
            market_data=market_data,
        )
    except ValueError:
        print(
            quote.option_type,
            quote.strike,
            quote.bid,
            quote.ask,
            quote.mid_price,
        )
        continue

call 110.0 222.4 225.7 224.05
call 115.0 211.05 214.75 212.9
call 120.0 212.45 215.75 214.1
call 125.0 207.4 210.75 209.075
call 130.0 202.4 205.95 204.175
call 135.0 191.1 194.85 192.975
call 140.0 192.55 195.75 194.15
call 150.0 182.2 185.85 184.02499999999998
call 155.0 177.1 181.0 179.05
call 160.0 172.6 175.9 174.25
call 165.0 167.3 171.0 169.15
call 170.0 162.65 165.9 164.275
call 175.0 157.7 160.9 159.3
call 180.0 152.7 155.95 154.325
call 185.0 147.75 151.05 149.4
call 190.0 142.75 146.1 144.425
call 195.0 137.8 141.05 139.425
call 205.0 127.85 131.1 129.475
call 210.0 122.9 126.15 124.525
call 220.0 113.0 115.7 114.35
call 235.0 98.15 101.1 99.625


In [12]:
from optionanalytics.models.option import EuropeanOption
from optionanalytics.models.enums import OptionType
from optionanalytics.models.market import MarketData
from optionanalytics.pricing.black_scholes import black_scholes

option = EuropeanOption(
    option_type=OptionType.CALL,
    strike=110,
    maturity=0.09315068493150686,
)

market_low = MarketData(
    spot=333.74,
    risk_free_rate=0.0418,
    volatility=1e-6,
)

market_high = MarketData(
    spot=333.74,
    risk_free_rate=0.0418,
    volatility=5.0,
)

print("BS(min vol):", black_scholes(option, market_low))
print("BS(max vol):", black_scholes(option, market_high))
print("Market:", 224.05)

BS(min vol): 224.16747408161828
BS(max vol): 257.7969096335238
Market: 224.05


In [14]:
from optionanalytics.cleaning.arbitrage import filter_price_bound_violations

valuation_date = datetime.date(2026,7,18)
cleanest_chain = filter_price_bound_violations(clean_chain, market_data, valuation_date)
smile = build_smile(cleanest_chain, market_data, valuation_date)

In [17]:
smile.underlying, smile.expiry, smile.points[0:5]

('AAPL',
 datetime.date(2026, 8, 21),
 [VolatilityPoint(strike=200.0, implied_volatility=0.7733219814787088),
  VolatilityPoint(strike=215.0, implied_volatility=0.5059425459785707),
  VolatilityPoint(strike=225.0, implied_volatility=0.5911053423362373),
  VolatilityPoint(strike=230.0, implied_volatility=0.5252843206663623),
  VolatilityPoint(strike=240.0, implied_volatility=0.48595244512146235)])